# Phase 1 (GitHub + Read + Q&A)

In [9]:
from dotenv import load_dotenv
import os
from agents.mcp import MCPServerStreamableHttp, MCPServerStdio
from agents import Agent, trace, Runner
from IPython.display import display, Markdown
import json

In [10]:
load_dotenv(override=True)

GITHUB_PAT = os.getenv("GITHUB_PAT")

### Defining MCP servers

In [11]:
github_params = {"url": "https://api.githubcopilot.com/mcp/",
        "headers": {
            "Authorization": f"Bearer {GITHUB_PAT}"
        }
    }

memory_params = {"command": "npx","args": ["-y", "@modelcontextprotocol/server-memory"],"env": {"MEMORY_FILE_PATH": "/Users/manuyadav/projects/coding_agent/src/Memory/memory.jsonl"}}

In [12]:
github_mcp_server = MCPServerStreamableHttp(params=github_params, client_session_timeout_seconds=30)
memory_mcp_server = MCPServerStdio(params=memory_params, client_session_timeout_seconds=30)

await github_mcp_server.connect()
await memory_mcp_server.connect()

In [13]:
github_instructions = "You are smart agent which works on github tools." 
memory_instructions = "You have to use your memory tools to store the information about the repository. This has to be used everytime, whenever we talks about repo."

instructions = f"""{github_instructions}

{memory_instructions}
"""

### Defining Agents

In [18]:
github_agent = Agent (
	name="Github agent",
	instructions=instructions,
	model="gpt-4.1-mini",
	mcp_servers=[github_mcp_server, memory_mcp_server]
)

with trace("Github test"):
	result = await Runner.run(github_agent, "Can you fetch list of my repo")

display(Markdown(result.final_output))

### Implementing Memory

In [20]:

MEMORY_FILE = "/Users/manuyadav/projects/coding_agent/src/Memory/memory.json"

def load_memory():
    if not os.path.exists(MEMORY_FILE):
        return {
            "chat_history": [],
            # "repo_context": {}
        }
    with open(MEMORY_FILE, "r") as f:
        return json.load(f)

def save_memory(memory):
    with open(MEMORY_FILE, "w") as f:
        json.dump(memory, f, indent=2)

In [21]:
def build_context(memory, user_input):
    chat_history = memory["chat_history"][-5:]

    context = ""

    for msg in chat_history:
        context += f"{msg['role']}: {msg['content']}\n"

    # Add repo ONLY when needed
   #  if intent in ["repo_query", "repo_task"]:
      #   context += f"\nRepo Context:\n{memory['repo_context']}\n"

    context += f"\nUser: {user_input}"

    return context

## Running our Agent

In [23]:
memory = load_memory()

user_input = "It is me, and i am looking ofr email reply one"

context = build_context(memory, user_input)

with trace("Coding agent"):
	result = await Runner.run(github_agent, context)

display(Markdown(result.final_output))

memory["chat_history"].append({"role": "user", "content": result.input})

memory["chat_history"].append({"role": "assistant", "content": result.final_output})

# Trim history
memory["chat_history"] = memory["chat_history"][-10:]

save_memory(memory)

Can you please provide the exact repository name where the "src" directory is located so I can access it and check what it does? Also, do you want me to look for email reply functionality specifically inside the "src" directory?